# Dominion: increase endpoint coverage

This notebook produces **one review table**. It does not assign confidence scores, confirm locations, create project routes, or write final project GeoJSON.

Use the state column only to choose the search area. Download all operators' substations, transmission lines, plants, and explicitly tagged switchyards. Switching substations are already included by `power=substation`.

The functions in `src/gridlock/location_discovery.py` use basic Python, pandas, requests, and regular expressions. Raw infrastructure geometries appear only as candidate geometry in the review CSV.

In [1]:
import pandas as pd

from gridlock.paths import PROCESSED_DATA_DIR
from gridlock.location_discovery import fetch_all_power, discover_locations

PROJECTS_PATH = PROCESSED_DATA_DIR / "dominion" / "dominion_projects.csv"
REVIEW_PATH = PROCESSED_DATA_DIR / "geo" / "dominion_location_review.csv"
df_projects = pd.read_csv(PROJECTS_PATH, dtype=str, keep_default_na=False)
if "state" not in df_projects:
    raise ValueError("The input must provide a state for every project")
df_projects["state"] = df_projects["state"].str.strip().str.upper()
projects = df_projects.to_dict("records")

## 1. Fetch all power infrastructure in each project state
`fetch_all_power(state)` reads the saved statewide responses when available. Otherwise it makes bulk Overpass requests and saves successful responses under `data/raw/osm/<STATE>/`.

There is no utility/operator filter. Names and aliases from both substations and transmission lines are used. Plant names can help locate endpoints that refer to generating stations. No project-by-project Overpass calls are made.

In [2]:
infrastructure = {}
for state, state_projects in df_projects.groupby("state"):
    infrastructure[state] = fetch_all_power(state)

## 2. Extract names, match locally, then search missing endpoints
`extract_endpoint_names()` reads both the title and description. It collects capitalized place phrases and removes voltage labels, construction verbs, and electrical terms. `normalize_endpoint()` handles punctuation, dashes, spacing, and common abbreviations such as St./Saint and Jct./Junction.

`match_endpoint()` first tries exact normalized names, then whole-word containment. It keeps all matching candidates rather than assigning a score. A named line can match an endpoint mentioned in its name. These are **review candidates**, not verified endpoints: for example a longer substation name containing a town name may be unrelated to the project.

For names still missing, `search_nominatim()` searches for a substation, then a power plant. Each unique state/name search is cached and reused across projects. Only returned power features with the matching state and name are counted; a town or street with the same name is not treated as a substation.

**Nominatim restrictions:** this is a small, user-requested, one-time fallback, on one machine and one thread. Every result is cached, requests identify GridLock, and the standard-library `time.sleep(1.1)` pauses between uncached requests. Do not schedule repeated bulk runs. [Read the public usage policy](https://operations.osmfoundation.org/policies/nominatim/). OSM data: ? OpenStreetMap contributors, [ODbL attribution](https://www.openstreetmap.org/copyright).

Failed requests stop the run instead of being counted as no match. Rerunning reuses earlier successful searches.

In [3]:
review, summary = discover_locations(projects, infrastructure, use_nominatim=True)

## 3. Save the single review table and print coverage
The table has exactly the requested eight columns. Each candidate gets a row. Unmatched endpoint names remain as rows with empty feature/geometry/source cells. Repeated project IDs are retained with their project names. Geometry is the source OSM feature, never a newly inferred project route.

Counts refer to distinct extracted endpoint names **within each project**, not the number of candidate rows. A shared endpoint can therefore count for multiple projects. ?Projects with two matched endpoints? means two or more, since some projects mention several sites. All counts describe candidate coverage, not confirmed locations.

No other processed output is generated by this notebook.

In [4]:
REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
review.to_csv(REVIEW_PATH, index=False)
for label, count in summary.items():
    print(f"{label}: {count}")

Total projects: 54
Total endpoint names extracted: 137
Endpoints matched: 67
Projects with at least one matched endpoint: 45
Projects with two matched endpoints: 18
Projects with no matched endpoints: 9
